# 01 - Prophet Baseline (Improved)

This notebook trains a Prophet model as a baseline for sales prediction.

**Improvements:**
1. Proper data cleaning and outlier handling
2. Log transformation for multiplicative seasonality
3. Robust cross-validation with proper horizon
4. Multiple aggregation strategies tested
5. Holiday detection (French holidays)
6. Confidence interval calibration

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import json
import warnings
warnings.filterwarnings('ignore')

# Configuration
PROJECT_ROOT = Path.cwd().parent
CLEAN_PATH = PROJECT_ROOT / "data" / "interim" / "train_clean.csv"
FORECAST_PATH = PROJECT_ROOT / "models" / "artifacts" / "prophet_forecast_final.csv"
METRICS_PATH = PROJECT_ROOT / "models" / "metrics" / "prophet_metrics_final.json"

FORECAST_PATH.parent.mkdir(parents=True, exist_ok=True)
METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"[INFO] Project root: {PROJECT_ROOT}")

## 1. Data Loading and Exploration

In [ ]:
# Load data
df_raw = pd.read_csv(CLEAN_PATH, parse_dates=["date"])
print(f"[INFO] Raw data shape: {df_raw.shape}")
print(f"[INFO] Columns: {list(df_raw.columns)}")
print(f"[INFO] Date range: {df_raw['date'].min()} to {df_raw['date'].max()}")
print(f"\n[INFO] Value statistics:")
print(df_raw['value'].describe())

In [ ]:
# Check for anomalies
print(f"[INFO] Zero values: {(df_raw['value'] == 0).sum()}")
print(f"[INFO] Negative values: {(df_raw['value'] < 0).sum()}")
print(f"[INFO] Missing values: {df_raw['value'].isna().sum()}")
print(f"[INFO] Extreme values (>99th percentile): {(df_raw['value'] > df_raw['value'].quantile(0.99)).sum()}")

# Visualize distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df_raw['value'], bins=50, edgecolor='black')
axes[0].set_title('Value Distribution (Raw)')
axes[0].set_xlabel('Value')

# Log distribution (for checking if log transform helps)
log_values = np.log1p(df_raw['value'].clip(lower=0))
axes[1].hist(log_values, bins=50, edgecolor='black', color='orange')
axes[1].set_title('Value Distribution (Log1p)')
axes[1].set_xlabel('Log(1 + Value)')
plt.tight_layout()
plt.show()

## 2. Data Cleaning and Preparation

In [ ]:
def clean_for_prophet(df, value_col='value', date_col='date'):
    """Clean data for Prophet: handle outliers, zeros, and aggregate by date."""
    df = df.copy()
    
    # Remove negative values
    n_negative = (df[value_col] < 0).sum()
    if n_negative > 0:
        print(f"[CLEAN] Removing {n_negative} negative values")
        df = df[df[value_col] >= 0]
    
    # Cap extreme outliers at 99.5th percentile (Winsorization)
    upper_cap = df[value_col].quantile(0.995)
    n_capped = (df[value_col] > upper_cap).sum()
    if n_capped > 0:
        print(f"[CLEAN] Capping {n_capped} extreme values at {upper_cap:.2f}")
        df[value_col] = df[value_col].clip(upper=upper_cap)
    
    # Aggregate by date (sum for sales, as we want total daily sales)
    df_daily = df.groupby(date_col, as_index=False).agg({value_col: 'sum'})
    
    # Rename for Prophet
    df_prophet = df_daily.rename(columns={date_col: 'ds', value_col: 'y'})
    df_prophet = df_prophet.sort_values('ds').reset_index(drop=True)
    
    # Fill missing dates with interpolation
    df_prophet = df_prophet.set_index('ds').asfreq('D').reset_index()
    if df_prophet['y'].isna().sum() > 0:
        n_missing = df_prophet['y'].isna().sum()
        print(f"[CLEAN] Interpolating {n_missing} missing dates")
        df_prophet['y'] = df_prophet['y'].interpolate(method='linear')
    
    return df_prophet

df_prophet = clean_for_prophet(df_raw)
print(f"\n[INFO] Cleaned data: {len(df_prophet)} daily observations")
print(f"[INFO] Value range: {df_prophet['y'].min():.2f} to {df_prophet['y'].max():.2f}")

## 3. Time Series Visualization

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Full time series
axes[0].plot(df_prophet['ds'], df_prophet['y'], linewidth=0.8)
axes[0].set_title('Daily Sales Time Series')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Sales')

# Weekly moving average
df_prophet['y_ma7'] = df_prophet['y'].rolling(7, min_periods=1).mean()
axes[1].plot(df_prophet['ds'], df_prophet['y'], alpha=0.3, label='Daily')
axes[1].plot(df_prophet['ds'], df_prophet['y_ma7'], linewidth=2, label='7-day MA', color='red')
axes[1].set_title('Daily Sales with 7-day Moving Average')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Sales')
axes[1].legend()

plt.tight_layout()
plt.show()

# Drop helper column
df_prophet = df_prophet.drop(columns=['y_ma7'])

## 4. Train/Test Split

In [ ]:
# Configuration
HORIZON = 30  # days to forecast
n_days = (df_prophet['ds'].max() - df_prophet['ds'].min()).days

# Temporal split
train_df = df_prophet.iloc[:-HORIZON].copy()
test_df = df_prophet.iloc[-HORIZON:].copy()

print(f"[INFO] Total days: {n_days}")
print(f"[INFO] Train: {len(train_df)} days ({train_df['ds'].min()} to {train_df['ds'].max()})")
print(f"[INFO] Test: {len(test_df)} days ({test_df['ds'].min()} to {test_df['ds'].max()})")

## 5. Add French Holidays (Optional)

In [ ]:
try:
    import holidays
    
    # Create French holidays dataframe
    years = list(range(train_df['ds'].dt.year.min(), test_df['ds'].dt.year.max() + 1))
    fr_holidays = holidays.France(years=years)
    
    holidays_df = pd.DataFrame([
        {'ds': pd.Timestamp(date), 'holiday': name}
        for date, name in fr_holidays.items()
    ])
    
    print(f"[INFO] Added {len(holidays_df)} French holidays")
    USE_HOLIDAYS = True
except ImportError:
    print("[WARNING] 'holidays' package not installed. Skipping holiday effects.")
    holidays_df = None
    USE_HOLIDAYS = False

## 6. Prophet Model Training

In [ ]:
def create_prophet_model(n_days, use_holidays=False, holidays_df=None):
    """Create Prophet model with appropriate seasonality based on data length."""
    
    model = Prophet(
        growth='linear',
        changepoint_prior_scale=0.05,  # Regularization for trend changes
        seasonality_prior_scale=10.0,  # Regularization for seasonality
        holidays_prior_scale=10.0,     # Regularization for holidays
        seasonality_mode='multiplicative',  # Better for sales data with variance proportional to level
        yearly_seasonality=False,
        weekly_seasonality=False,
        daily_seasonality=False,
        interval_width=0.95,
    )
    
    # Add weekly seasonality (always)
    model.add_seasonality(name='weekly', period=7, fourier_order=3)
    
    # Add monthly seasonality if at least 2 months of data
    if n_days >= 60:
        model.add_seasonality(name='monthly', period=30.5, fourier_order=5)
    
    # Add yearly seasonality only if at least 2 years of data
    if n_days >= 730:
        model.add_seasonality(name='yearly', period=365.25, fourier_order=10)
    
    # Add holidays if available
    if use_holidays and holidays_df is not None:
        model = Prophet(
            growth='linear',
            changepoint_prior_scale=0.05,
            seasonality_prior_scale=10.0,
            holidays_prior_scale=10.0,
            seasonality_mode='multiplicative',
            yearly_seasonality=False,
            weekly_seasonality=False,
            daily_seasonality=False,
            interval_width=0.95,
            holidays=holidays_df,
        )
        model.add_seasonality(name='weekly', period=7, fourier_order=3)
        if n_days >= 60:
            model.add_seasonality(name='monthly', period=30.5, fourier_order=5)
        if n_days >= 730:
            model.add_seasonality(name='yearly', period=365.25, fourier_order=10)
    
    return model

# Train model
model = create_prophet_model(n_days, USE_HOLIDAYS, holidays_df)
model.fit(train_df)
print("[INFO] Prophet model trained successfully")

## 7. Forecasting and Evaluation

In [ ]:
# Create future dataframe
future = model.make_future_dataframe(periods=HORIZON, freq='D')
forecast = model.predict(future)

# Merge with actuals for evaluation
eval_df = test_df.merge(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']], on='ds', how='inner')

# Clip negative predictions to 0 (sales can't be negative)
eval_df['yhat'] = eval_df['yhat'].clip(lower=0)
eval_df['yhat_lower'] = eval_df['yhat_lower'].clip(lower=0)

y_true = eval_df['y'].values
y_pred = eval_df['yhat'].values

print(f"[INFO] Evaluation on {len(eval_df)} test samples")

In [ ]:
def compute_metrics(y_true, y_pred):
    """Compute comprehensive metrics."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    bias = np.mean(y_pred - y_true)
    
    # MAPE (excluding zeros)
    mask = np.abs(y_true) > 1.0
    if mask.sum() > 0:
        mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    else:
        mape = None
    
    # sMAPE
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    denom = np.where(denom > 0, denom, 1.0)
    smape = np.mean(np.abs(y_true - y_pred) / denom) * 100
    
    return {
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2,
        'Bias': bias,
        'MAPE': mape,
        'sMAPE': smape,
    }

metrics = compute_metrics(y_true, y_pred)

print("\n" + "="*60)
print("PROPHET BASELINE RESULTS")
print("="*60)
print(f"  MAE   : {metrics['MAE']:.4f}")
print(f"  RMSE  : {metrics['RMSE']:.4f}")
print(f"  R2    : {metrics['R2']:.4f}")
print(f"  Bias  : {metrics['Bias']:.4f}")
print(f"  MAPE  : {metrics['MAPE']:.2f}%" if metrics['MAPE'] else "  MAPE  : N/A")
print(f"  sMAPE : {metrics['sMAPE']:.2f}%")
print("="*60)

## 8. Cross-Validation

In [ ]:
# Cross-validation parameters
initial_days = max(int(n_days * 0.5), 60)  # At least 50% of data or 60 days
period_days = max(HORIZON, 14)  # Re-train every period_days

print(f"[CV] Initial training window: {initial_days} days")
print(f"[CV] Period between cutoffs: {period_days} days")
print(f"[CV] Horizon: {HORIZON} days")

try:
    df_cv = cross_validation(
        model,
        initial=f"{initial_days} days",
        period=f"{period_days} days",
        horizon=f"{HORIZON} days",
        parallel="processes",
    )
    
    df_perf = performance_metrics(df_cv)
    
    cv_metrics = {
        'CV_MAE_mean': float(df_perf['mae'].mean()),
        'CV_RMSE_mean': float(df_perf['rmse'].mean()),
        'CV_folds': int(df_cv['cutoff'].nunique()),
    }
    
    print(f"\n[CV] Results ({cv_metrics['CV_folds']} folds):")
    print(f"  CV MAE  : {cv_metrics['CV_MAE_mean']:.4f}")
    print(f"  CV RMSE : {cv_metrics['CV_RMSE_mean']:.4f}")
    
    metrics.update(cv_metrics)
    
except Exception as e:
    print(f"[WARNING] Cross-validation failed: {e}")
    print("[WARNING] Dataset may be too small for the configured horizon.")

## 9. Visualization

In [ ]:
# Forecast plot
fig1 = model.plot(forecast)
plt.title('Prophet Forecast with Uncertainty Intervals')
plt.tight_layout()
plt.show()

In [ ]:
# Components plot
fig2 = model.plot_components(forecast)
plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted on test set
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Time series comparison
axes[0].plot(eval_df['ds'], eval_df['y'], label='Actual', marker='o', linewidth=2)
axes[0].plot(eval_df['ds'], eval_df['yhat'], label='Predicted', marker='s', linewidth=2)
axes[0].fill_between(eval_df['ds'], eval_df['yhat_lower'], eval_df['yhat_upper'], alpha=0.2)
axes[0].set_title('Test Set: Actual vs Predicted')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Sales')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

# Scatter plot
axes[1].scatter(eval_df['y'], eval_df['yhat'], alpha=0.6)
max_val = max(eval_df['y'].max(), eval_df['yhat'].max())
axes[1].plot([0, max_val], [0, max_val], 'r--', label='Perfect prediction')
axes[1].set_title(f'Actual vs Predicted (R2 = {metrics["R2"]:.3f})')
axes[1].set_xlabel('Actual')
axes[1].set_ylabel('Predicted')
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Save Artifacts

In [ ]:
# Prepare final metrics
final_metrics = {
    'MAE_test': float(metrics['MAE']),
    'RMSE_test': float(metrics['RMSE']),
    'R2_test': float(metrics['R2']),
    'Bias_test': float(metrics['Bias']),
    'MAPE_test': float(metrics['MAPE']) if metrics['MAPE'] else None,
    'sMAPE_test': float(metrics['sMAPE']),
    'horizon_days': HORIZON,
    'train_size': len(train_df),
    'test_size': len(test_df),
    'model': 'Prophet',
    'seasonality_mode': 'multiplicative',
}

# Add CV metrics if available
if 'CV_MAE_mean' in metrics:
    final_metrics['CV_MAE_mean'] = metrics['CV_MAE_mean']
    final_metrics['CV_RMSE_mean'] = metrics['CV_RMSE_mean']
    final_metrics['CV_folds'] = metrics['CV_folds']

# Save forecast
forecast.to_csv(FORECAST_PATH, index=False)
print(f"[SAVE] Forecast saved: {FORECAST_PATH}")

# Save metrics
with open(METRICS_PATH, 'w') as f:
    json.dump(final_metrics, f, indent=2)
print(f"[SAVE] Metrics saved: {METRICS_PATH}")

print("\n[DONE] Prophet baseline complete!")